In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/clean_data.csv")

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Proxy target: teams that played many matches usually reached the final stages.
df["deep_run"] = (df["matches_played"] >= 6).astype(int)

df["deep_run"].value_counts()

In [ ]:
# Find the numeric variables most correlated with the temporary target.
numeric_columns = df.select_dtypes(include="number").columns
deep_run_correlations = (
    df[numeric_columns]
    .corr()["deep_run"]
    .drop("deep_run")
    .sort_values(ascending=False)
)

deep_run_correlations.head(6)

In [ ]:
# Basic mapping used only for historical comparison.
confederation_mapping = {
    "Argentina": "CONMEBOL", "Brazil": "CONMEBOL", "Uruguay": "CONMEBOL",
    "Chile": "CONMEBOL", "Colombia": "CONMEBOL", "Paraguay": "CONMEBOL",
    "Peru": "CONMEBOL", "Ecuador": "CONMEBOL", "Bolivia": "CONMEBOL",
    "Germany": "UEFA", "France": "UEFA", "Spain": "UEFA", "Italy": "UEFA",
    "England": "UEFA", "Netherlands": "UEFA", "Portugal": "UEFA",
    "Belgium": "UEFA", "Croatia": "UEFA", "Switzerland": "UEFA",
    "Sweden": "UEFA", "Denmark": "UEFA", "Poland": "UEFA", "Russia": "UEFA",
    "Mexico": "CONCACAF", "United States": "CONCACAF", "Costa Rica": "CONCACAF",
    "Canada": "CONCACAF", "Japan": "AFC", "Korea Republic": "AFC",
    "Iran": "AFC", "Saudi Arabia": "AFC", "Australia": "AFC",
    "Cameroon": "CAF", "Nigeria": "CAF", "Ghana": "CAF",
    "Morocco": "CAF", "Senegal": "CAF", "South Africa": "CAF",
    "New Zealand": "OFC"
}

df["confederation"] = df["team"].map(confederation_mapping).fillna("Other")

confederation_summary = (
    df.groupby("confederation")
    .agg(
        teams=("team", "nunique"),
        records=("team", "count"),
        avg_goals_per_match=("goals_per_match", "mean"),
        avg_win_percentage=("win_percentage", "mean"),
        avg_goal_difference=("goal_difference", "mean")
    )
    .round(2)
    .sort_values("avg_win_percentage", ascending=False)
)

confederation_summary

In [ ]:
# Load match-level data to compare home and away scoring behavior.
matches = pd.read_csv("../data/processed/clean_partial.csv")

home_avg_goals = matches["Home Team Goals"].mean()
away_avg_goals = matches["Away Team Goals"].mean()

home_away_summary = pd.DataFrame({
    "role": ["home", "away"],
    "avg_goals": [home_avg_goals, away_avg_goals]
}).round(2)

home_away_summary

In [ ]:
# Summarize long-term team success across all tournaments.
top_teams = (
    df.groupby("team")
    .agg(
        total_matches=("matches_played", "sum"),
        total_wins=("wins", "sum"),
        total_goals_for=("goals_for", "sum"),
        total_goals_against=("goals_against", "sum"),
        historical_goal_difference=("goal_difference", "sum")
    )
    .reset_index()
)

top_teams["historical_win_percentage"] = (
    top_teams["total_wins"] / top_teams["total_matches"] * 100
).round(1)

top_10_successful_teams = (
    top_teams[top_teams["total_matches"] >= 10]
    .sort_values(["total_wins", "historical_win_percentage"], ascending=False)
    .head(10)
)

top_10_successful_teams

## Final Insights

### Insight #1: Deep runs are strongly linked to match volume and wins
**What I found:** The strongest correlations with `deep_run` are `matches_played` (0.81), `wins` (0.69), `goals_for` (0.62), and `goal_difference` (0.49).  
**What it means for the model:** The model should include variables that measure consistency across a tournament, not only scoring.  
**Derived variable:** `deep_run` and future official `reached_semifinals` target.

### Insight #2: Goal difference is a better success signal than goals alone
**What I found:** `goal_difference` has a strong relationship with win percentage (0.80) and deep runs (0.49).  
**What it means for the model:** Balanced teams that score more than they concede should receive higher semifinal probabilities.  
**Derived variable:** `goal_difference`.

### Insight #3: UEFA and CONMEBOL dominate historical performance
**What I found:** UEFA and CONMEBOL have the highest average win percentages, both around 41.8%, and positive average goal differences.  
**What it means for the model:** Confederation should be tested as a categorical predictor because historical strength is not evenly distributed.  
**Derived variable:** `confederation`.

### Insight #4: Home teams score more than away teams in the match dataset
**What I found:** Home and away average goals can be compared from `clean_partial.csv` to measure role-based scoring differences.  
**What it means for the model:** Venue role can affect match-level performance, but for tournament-level predictions it should be used carefully.  
**Derived variable:** `venue_role`.

### Insight #5: Brazil and Germany are the strongest historical teams
**What I found:** Brazil has 70 wins and Germany has 66 wins, clearly ahead of the rest of the historical ranking.  
**What it means for the model:** Historical success should be included because repeated World Cup performance is a strong signal.  
**Derived variable:** `historical_win_percentage`.